# Lab 1 — From Raw Data to a First Model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/INCORTX/INCORTX.github.io/blob/master/DataAnalytics/session-01/lab_01.ipynb)

**Data Analytics · DTii 1/2026 · Lab session 1** · about 180 minutes

### By the end of this session you will be able to
1. **See where a dataset is dirty** and decide what to do about it, with a reason you can defend
2. **Write a `Pipeline` with no data leakage** — the most common mistake in this work, and the one you cannot spot by eye
3. **Judge whether a model is actually any good**, by comparing it to a baseline and choosing a metric that fits the problem

### How the session runs

| | Part | What happens |
|---|---|---|
| **1** | 🎬 **Demo — 45 min** | Everything down to the topic list. The instructor walks it; you watch. Do not type along — you will get this file to keep. |
| **2** | 📋 **Pick a topic** | Your group claims one of the twenty. First come, first served. |
| **3** | 🟠 **Your hour — 60 min** | The section at the bottom. Five steps, your own data, your own notebook. |
| **4** | 🎤 **Show it — 60 min** | Every group puts their notebook on screen for six minutes. |

| Mark | Meaning |
|---|---|
| 🔵 **Demo** | Part of what the instructor walks |
| 🔍 **Try later** | Not done in class — worth doing when you reread this |
| ✅ **Expected** | What the cell should print, so you can check yourself |
| 💥 **Meant to fail** | Read the error until you understand it, then move on |

> **The bottom section runs on its own.** It re-imports everything and loads your own data, so a demo
> cell that failed on a slow connection cannot take your hour down with it.

> **Every dataset here loads in one line.** No file uploads, no Google Drive, no paths to fix.
> Chart text is in English so the figures render the same on every machine.

---
## 🎤 The 45 minutes we actually walk through

**This notebook holds about 73 minutes of material and the demo slot is 45.** That is on purpose —
it is a reference you keep, not a script read start to finish.

**These eleven we do together.** Everything else is yours to read when you need it.

| | Walked in the demo | Why this one earns the time |
|:--:|---|---|
| 1 | **How the session runs** | so nobody spends the work hour guessing what to hand in |
| 2 | **A.1 · A.2** — the data, and a model in three columns | the shortest path from a CSV to a number |
| 3 | **A.3** — the baseline | **the number every other number in this course is read against** |
| 4 | **A.5** — then the real world arrives | why the clean version was a lie |
| 5 | **A.6** — five ways this data is broken | the checklist you will run on your own topic in 30 minutes |
| 6 | **C.3** — the trap that makes you happy for ten seconds | **100% accuracy from a column that is the answer** — reading about leakage is not the same as watching it |
| 7 | **C.6** — where accuracy lies to your face | the metric argument, with the numbers in front of you |
| 8 | **C.8** — `Pipeline`, and why it makes C.7 impossible | the one habit that prevents the most common zero |
| 9 | **D.2** — does a bigger model help? | it does not, and the table says so |
| 10 | **D.5** — how is the model wrong? | where a confusion matrix stops being decoration |
| 11 | **Part 2 — picking your topic** | you skim and claim — this is not read out loud |

**≈ 36 minutes**, which leaves room for the questions that always come. The rest — B's four file
shapes, C.1 · C.2 · C.3.1 · C.7, D's reference blocks, and the worked blocks marked 🔎 — is
**reference for your hour and your homework.**

> **Four things are deliberately not in this session, and none of them are gone from the course.**
>
> | Not here | Where it lands, and why there |
> |---|---|
> | **Regression** — predicting a number | **session 2**, whose lecture covers it. Teaching MAE and R² now would be ahead of the theory |
> | **Outliers, IQR and box plots** | **session 2**. Trees barely notice outliers, so here it would be a fact with no consequence — and session 2's lecture reaches outliers through residuals |
> | **Feature scaling** | **session 3**. This session is trees, which ignore scale. In session 3, k-Means gives you the wrong answer without it |
> | **Zips and awkward files** | **session 2 onward**, when the topic list arrives a week early and you load before class rather than inside the marked hour |

> **The 🔎 blocks are worked examples with every answer filled in.** They are there for when you are
> stuck on your own data and want to see the same step done end to end on something else.

### 📖 Before you start — if pandas is not yet second nature

**There is no preparation class before this one**, so this is it. Nothing here is taught in the
session; it is the handful of things you will reach for during your hour, in one place. Skim it now,
come back to it when you are stuck.

| You want to | You write |
|---|---|
| open a file | `df = pd.read_csv(url)` |
| how big is it | `df.shape` → `(rows, columns)` |
| look at it | `df.head()` · `df.sample(5)` |
| what type is each column | `df.info()` |
| numbers at a glance | `df.describe()` |
| **what is missing** | `df.isna().sum()` |
| how many of each label | `df['col'].value_counts()` |
| one column | `df['col']` · several: `df[['a','b']]` |
| rows that match | `df[df['age'] > 30]` |
| drop a column | `df.drop(columns=['col'])` |
| average per group | `df.groupby('label').mean()` |
| a quick chart | `df['col'].hist()` · `df.plot.scatter(x='a', y='b')` |

> **The one that catches everyone:** `df.describe()` only shows numeric columns. A text column that
> should have been a number will silently not appear — which is itself the signal that something needs
> converting.

In [ ]:
# Run this to see all of the above on a real table. Nothing here is assessed.
import pandas as pd
demo = pd.read_csv('https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv')

print('shape       :', demo.shape)
print('missing     :', demo.isna().sum().sum(), 'cells')
print('species     :', demo['species'].value_counts().to_dict())
print()
print(demo.describe().round(2).to_string())

In [ ]:
# ── Setup: run this cell first ──────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

SEED = 42                      # every random step in this notebook uses this
np.random.seed(SEED)

pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 30)
plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('pandas      ', pd.__version__)
print('seaborn     ', sns.__version__)
import sklearn; print('scikit-learn', sklearn.__version__)

---
---
# 🎬 Part 1 — The Demo · blocks A to D

**Watch, do not type along.** Each block header says which subsections are walked live (🎤)
and which are reference (📖). The full 45-minute route is the table above.

---
# A · A Model in 15 Minutes — Then the Real World Arrives
🎤 **Walked live:** A.1 · A.2 · A.3 · A.5 · A.6 &nbsp;·&nbsp; 📖 **read on your own:** A.4

We build a working model **first**, and only then look at what breaks it. The order matters:
learn data cleaning before you have seen a model fail, and cleaning feels like chores nobody explained.

### 🔵 A.1 — The dataset we use all session

`titanic` holds 891 passenger records. The goal is to predict who **survived** (`survived` = 1) and who did not (`survived` = 0).

We chose it because it is **genuinely messy on its own** — none of the mess below was manufactured for teaching.

In [ ]:
df = sns.load_dataset('titanic')      # one line, no upload, no Drive
print('shape:', df.shape)
df.head()

### 🔵 A.2 — First model, three columns only

Start with three numeric columns that have **no missing values**, so the code runs at all:

| Column | Meaning |
|---|---|
| `pclass` | Ticket class: 1 / 2 / 3 |
| `sibsp` | Siblings and spouses aboard |
| `fare` | Ticket price |

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X3 = df[['pclass', 'sibsp', 'fare']]
y  = df['survived']

X_train, X_test, y_train, y_test = train_test_split(
    X3, y, test_size=0.2, random_state=SEED, stratify=y)

tree = DecisionTreeClassifier(random_state=SEED)
tree.fit(X_train, y_train)

acc = accuracy_score(y_test, tree.predict(X_test))
print(f'Accuracy: {acc:.4f}')

✅ **Expected:** `Accuracy: 0.6480`

Six lines and the model is right about 65% of the time. That sounds usable.

**But is it?** You cannot answer that yet — not until you know what *guessing without a model at all* would score.

### 🔵 A.3 — The line you have to beat (baseline)

Most passengers did not survive. So the dumbest possible model is: **say "did not survive" for everyone**, without looking at any data.

`DummyClassifier` does exactly that, and the number it produces is the line a real model has to clear.

In [ ]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)

base_acc = accuracy_score(y_test, baseline.predict(X_test))
print(f'Baseline accuracy: {base_acc:.4f}   <- guessing "did not survive" every time')
print(f'Our tree         : {acc:.4f}')
print(f'Improvement      : {acc - base_acc:+.4f}')

✅ **Expected:** baseline `0.6145` · tree `0.6480` · improvement `+0.0335`

**This is the most expensive lesson in the session.** That respectable-looking 65% is worth exactly
**3.4 points** more than guessing.

> **Rule to keep:** a model's number means nothing without something to compare it against.
> Every time you report a result, the baseline goes next to it.

### 🔵 A.4 — What did the model actually learn?

A decision tree can be **read as sentences**, which most other models cannot.

In [ ]:
from sklearn.tree import plot_tree

small_tree = DecisionTreeClassifier(max_depth=3, random_state=SEED).fit(X_train, y_train)
print(f'depth-3 accuracy: {accuracy_score(y_test, small_tree.predict(X_test)):.4f}')

plt.figure(figsize=(15, 6))
plot_tree(small_tree, feature_names=X3.columns, class_names=['died', 'survived'],
          filled=True, rounded=True, fontsize=9, impurity=False)
plt.title('What the tree learned (first 3 levels)')
plt.tight_layout(); plt.show()

✅ **Expected:** `depth-3 accuracy: 0.6425`, plus a tree you can read

Try reading the leftmost branch out loud, something like
*"if the fare is below X and the passenger is in third class, predict died."*

**Notice:** the three-level tree scores 0.6425 and the unlimited one scores 0.6480 — nearly the same,
despite being far more complex. Hold on to that observation; we come back to it in part D.

### 💥 A.5 — Then the real world arrives

We hand-picked three clean columns. On a real job nobody picks them for you.

**The next cell is meant to fail.** Throw the whole DataFrame at the same model and read the error.

In [ ]:
X_all = df.drop(columns=['survived'])

try:
    DecisionTreeClassifier(random_state=SEED).fit(X_all, y)
except ValueError as e:
    print('ValueError:', e)

✅ **Expected:** `ValueError: could not convert string to float: 'male'`

scikit-learn only takes numbers, so a column like `sex` holding `'male'` / `'female'` stops it dead.

That is **problem one**, and we are lucky with it, because it makes a noise. The rest are quieter.

In [ ]:
missing = df.isna().sum()
missing = missing[missing > 0].to_frame('missing_rows')
missing['percent'] = (missing['missing_rows'] / len(df) * 100).round(1)
print(missing)

✅ **Expected:**

| Column | Missing | % |
|---|---:|---:|
| `age` | 177 | 19.9 |
| `embarked` | 2 | 0.2 |
| `deck` | 688 | 77.2 |
| `embark_town` | 2 | 0.2 |

**Problem two:** `deck` is missing for **77%** of passengers. Filling it in is wrong, throwing it away wastes
what is there. There is no textbook answer — decisions like this are the actual daily work of a data analyst.

### 🔎 A.6 — Counting the ways this data is broken

Four commands, and every one of them finds something: `df.info()` · `df.describe()` ·
`df.nunique()` · `df.duplicated().sum()`.

**Read the output before the answer below it.** How many problems can you name?

In [ ]:
df.info()
print('\nduplicated rows:', df.duplicated().sum())
print('\nunique values per column:')
print(df.nunique().sort_values())
print('\nfare spread:')
print(df['fare'].describe())

**Five problems, from those four commands**

| | Problem | Where | What it costs you |
|:--:|---|---|---|
| 1 | **Five text columns** a model cannot take as they are | `sex` `embarked` `who` `embark_town` `alive` | `ValueError: could not convert string to float` |
| 2 | **Missing values, and not evenly** | `deck` 688 of 891 (77%) · `age` 177 (20%) · `embarked` 2 | a column three quarters empty is a different decision from one 20% empty |
| 3 | **107 duplicated rows** | whole rows | the same passenger counted twice in both training and scoring |
| 4 | **Columns that say the same thing twice** | `survived`/`alive` · `pclass`/`class` · `embarked`/`embark_town` | `alive` **is the answer in words** — keeping it is the leakage you meet in C.3 |
| 5 | **Fares that jump to extremes** | `fare` max 512.33 against a 75th percentile of 31.00 — **16.5×** | scaling and distance-based models are dragged by a handful of rows |

**Number 4 is the one worth remembering.** The other four announce themselves with an error or an odd
number. That one runs perfectly and gives you 100% accuracy, which is why C.3 spends time on it.

---
# B · Data Collection — Getting the Data In
📖 **All of B is reference.** Not walked in the demo — **you will need it the moment your topic's zip
does not open**, and the work hour has a one-cell recap of all four shapes.

Your group project starts from nothing, not from a DataFrame somebody prepared. If you cannot load data,
you do not have a project.

Your group project starts from a URL somebody else wrote, not from a DataFrame we prepared. Every one of
the twenty assignment topics arrives in **one of four shapes**, and this section is the code for all four.
Learn them once here and the topic list stops being a list of obstacles.

### 🔵 B.1 — A CSV that lives on the web

`read_csv` takes a URL directly. No downloading first.

In [ ]:
CSV_URL = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv'

penguins = pd.read_csv(CSV_URL)
print('shape:', penguins.shape)
penguins.head(3)

✅ **Expected:** `shape: (344, 7)`

pandas reads Excel and JSON the same way — `pd.read_excel(url)` · `pd.read_json(url)`.

### 🔵 B.2 — When it is a zip, which is most of the time

Most of UCI ships zipped, so the file you want is inside an archive you never save to disk.
Fetch the bytes, open them as an archive, and read the member you want straight out of it.

We demo on the iris set — deliberately **not** one of the twenty, so nobody's group work is done for them.

In [ ]:
import io as _io
import zipfile
import urllib.request

URL = 'https://archive.ics.uci.edu/static/public/53/iris.zip'

with urllib.request.urlopen(URL) as response:
    archive = zipfile.ZipFile(_io.BytesIO(response.read()))

print('files inside the archive:', archive.namelist())

✅ **Expected:** `['Index', 'bezdekIris.data', 'iris.data', 'iris.names']`

**Always print `namelist()` first.** The file you want is rarely the only one in there, and its name is
rarely what you guessed. The `.names` file is usually the documentation — read it, it tells you the columns.

In [ ]:
iris = pd.read_csv(archive.open('iris.data'), header=None,
                   names=['sepal_len', 'sepal_wid', 'petal_len', 'petal_wid', 'species'])
print('shape:', iris.shape)
iris.head(3)

✅ **Expected:** `shape: (150, 5)`

**`header=None` matters.** Without it pandas eats the first row of data and turns it into your column
names — you lose a record and every column is called something like `5.1`. Several of the twenty topics
have no header row; their entries say so.

### 🔵 B.3 — The four shapes, and which topics are which

| Shape | Code | Topics that arrive this way |
|---|---|---|
| **Library call** | `load_wine(as_frame=True)` · `load_dataset('penguins')` · `fetch_openml('adult', version=1, as_frame=True)` | most of session 1 |
| **CSV at a URL** | `pd.read_csv(url)` — add `names=[...]` if there is no header row | topic 1 |
| **Zip at a URL** | the cell above | session 2 onward |
| **Zip inside a zip** | open the outer, then wrap the inner member in `BytesIO` again | session 2 onward |

**One awkward case you will meet later:** a zip whose member is an `.xls`, not a CSV. That needs
`pip install xlrd` and `pd.read_excel(..., header=1)`, because the real column names sit on row two.

```python
# zip inside a zip
outer = zipfile.ZipFile(_io.BytesIO(response.read()))
inner = zipfile.ZipFile(_io.BytesIO(outer.read('bank.zip')))
df = pd.read_csv(inner.open('bank-full.csv'), sep=';')
```

> **Every topic in the list names its file and its separator**, because those two are what cost you
> twenty minutes when they are wrong. If a topic entry says `;`, use `sep=';'` — comma is not the default
> everywhere in the world.

### 🔵 B.4 — Three traps that show up in every real file

Files from Thai government portals hit all three of these regularly.

In [ ]:
from io import StringIO

raw = '''date,province,population,revenue
2026-01-15,Metro North,"5,494,932","1,240,500.75"
2026-02-15,Riverside,"1,798,120","320,900.00"'''

bad = pd.read_csv(StringIO(raw))
print('-- read naively --')
print(bad.dtypes)

good = pd.read_csv(StringIO(raw), thousands=',', parse_dates=['date'])
print('\n-- read with thousands= and parse_dates= --')
print(good.dtypes)
print()
print(good)

✅ **Expected:** the first read gives `population` and `revenue` as `object` (text) — you cannot do arithmetic on them.
The second gives `int64` / `float64`, and `date` as `datetime64`.

**The three traps:**
1. **Thousands separators.** `"5,494,932"` reads as text · fix with `thousands=','`
2. **Dates** read as text · fix with `parse_dates=[...]`
3. **Thai encoding.** Files from older systems are often `cp874`, not `utf-8`.
   If the characters come out garbled, try `pd.read_csv(path, encoding='cp874')`

### 🔎 B.5 — A load is not finished until you can say where it came from

Loading is three lines. **Recording what you loaded is the fourth**, and it is the one that gets
skipped — then slide 3 of the homework has nothing to put on it.

Every load in this course prints four things: **topic · source · licence · shape.**

In [ ]:
GROUP_TOPIC = 'example: car fuel efficiency'
SOURCE_URL  = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/mpg.csv'
LICENSE     = 'seaborn-data (BSD-3-Clause) - free to use'

mine = pd.read_csv(SOURCE_URL)
print('topic  :', GROUP_TOPIC)
print('source :', SOURCE_URL)
print('licence:', LICENSE)
print('shape  :', mine.shape)
mine.head()

✅ **Expected:** `shape (398, 9)` — 398 cars, 9 columns.

**Do the same for your own topic in the work hour** — the four lines above are what page 2 of your slide deck is made of.

---
# C · Data Preprocessing — Making the Data Usable
🎤 **Walked live:** C.3 leakage · C.6 imbalance · C.8 Pipeline &nbsp;·&nbsp; 📖 **read on your own:** C.1 · C.2 · C.3.1 · C.7

This is the longest part of the session because it is where the time actually goes in this job,
and where people go wrong without noticing.
**The most dangerous mistake is the one that makes your results look better.**

### 🔵 C.1 — Repeated rows, and the trap hiding in them

In [ ]:
print('rows flagged as duplicated:', df.duplicated().sum())
df[df.duplicated(keep=False)].sort_values(['pclass', 'sex', 'age']).head(6)

✅ **Expected:** `rows flagged as duplicated: 107`

**Do not reach for `drop_duplicates()` yet.**

The Titanic carried 891 distinct passengers. `duplicated()` flags 107 rows because this dataset
**has no identifier column**, and plenty of third-class men happen to match on every single field.

> **Rule:** `duplicated()` tells you *the values repeat*. It does not tell you *it is the same person*.
> Delete here and you delete 107 real people. Before dropping duplicates, always ask
> **what makes one row different from another** in this dataset.

### 🔵 C.2 — Missing values: `deck` is 77% empty, now what?

There is no right answer, only a decision you can defend:

| Option | What you gain | What you lose |
|---|---|---|
| **Drop the column** | Nothing to guess at | The 203 passengers who do have a value |
| **Drop the rows** | Everything left is complete | **203 rows out of 891** — you threw away 77% of the data |
| **Fill with the most common value** | Every row survives | 688 rows get a value you invented = a fake signal |
| **Turn it into "cabin known / unknown"** | The missingness itself becomes data | Still a guess that missingness means something |

**What we will do:** drop `deck` — because the 77% is not missing at random.
Third-class passengers almost never had a cabin recorded, so filling it in means inventing a story
about the largest group in the data.

In [ ]:
print('deck present by class:')
print(df.groupby('pclass')['deck'].apply(lambda s: f'{s.notna().sum():3d} / {len(s):3d}'))

✅ **Expected:** class 1 has 175/216 · class 2 has 16/184 · class 3 has 12/491

That is the evidence that the missingness is not random — which changes the answer to
"should we fill it in?" completely.

### 💥 C.3 — The trap that will make you happy for the wrong reason

Now let us deal with the text columns. The fastest way is `pd.get_dummies()`, which turns
everything into numbers in one go.

**Run it and look hard at the number.**

In [ ]:
from sklearn.model_selection import train_test_split

naive = pd.get_dummies(df.drop(columns=['survived'])).fillna(0)

Xn_tr, Xn_te, yn_tr, yn_te = train_test_split(
    naive, y, test_size=0.2, random_state=SEED, stratify=y)

naive_model = DecisionTreeClassifier(random_state=SEED).fit(Xn_tr, yn_tr)
print(f'Accuracy: {accuracy_score(yn_te, naive_model.predict(Xn_te)):.4f}')

✅ **Expected:** `Accuracy: 1.0000`

**Every single prediction correct.** If that feels good, stop and get suspicious instead.

No real problem is answered perfectly. **A result that is too good is almost always a sign something is wrong.**

In [ ]:
importance = pd.Series(naive_model.feature_importances_, index=naive.columns)
print(importance.sort_values(ascending=False).head(5).round(4))

✅ **Expected:** `alive_no  1.0`, and every other feature at `0.0`

The model learned nothing. It found **the answer sitting in the input.**

In [ ]:
print(pd.crosstab(df['survived'], df['alive']))

✅ **Expected:** a perfect diagonal — `survived=0` with `alive='no'` 549 times, `survived=1` with `alive='yes'` 342 times

The `alive` column **is the target, spelled as words.**

> ### 🎯 Target leakage
> Target leakage is when the input contains something you could only know **after** you know the answer.
> The model scores beautifully in testing and then **fails completely in production**, because that column
> is not there when you need it.
>
> Symptoms to suspect immediately: unusually high accuracy, or a single feature holding almost all the importance.

### 🔵 C.3.1 — Which columns to remove, and why each one

| Column | Reason |
|---|---|
| `alive` | **The answer itself** — full leakage |
| `class` | Identical to `pclass` in every row (First/Second/Third ↔ 1/2/3) |
| `who` · `adult_male` | Derived from `sex` + `age`, which we already have — no new information |
| `deck` | 77% missing, and not at random (see C.2) |

Redundant columns are not as harmful as leakage, but they make `feature_importances_` hard to read,
because the importance gets split between columns that say the same thing.

In [ ]:
LEAKY_OR_DUPLICATE = ['alive', 'class', 'who', 'adult_male', 'deck']

data = df.drop(columns=LEAKY_OR_DUPLICATE)
X = data.drop(columns=['survived'])
y = data['survived']

print('columns kept:', list(X.columns))

✅ **Expected:** `['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'embark_town', 'alone']`

### 🔵 C.6 — Imbalance: where accuracy lies to your face

Titanic is 62:38, which is not skewed enough to show the problem.
So we will **randomly drop survivors until they are 10% of the data, purely as a demonstration**
(this version is deliberately distorted — it is not real data, it is here to show the effect).

In [ ]:
from sklearn.metrics import f1_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

NUM = ['age', 'sibsp', 'parch', 'fare', 'pclass']   # C.6 is now the first block that needs these
CAT = ['sex', 'embarked', 'embark_town', 'alone']

died = data[data.survived == 0]
kept = data[data.survived == 1].sample(n=int(len(died) * 0.10 / 0.90), random_state=SEED)
imb  = pd.concat([died, kept]).sample(frac=1, random_state=SEED)
print('class balance now:', imb.survived.value_counts(normalize=True).round(3).to_dict())

Xi, yi = imb.drop(columns=['survived'])[NUM], imb['survived']
Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(Xi, yi, test_size=0.2, random_state=SEED, stratify=yi)

dumb = DummyClassifier(strategy='most_frequent').fit(Xi_tr, yi_tr)
tree_i = Pipeline([('imp', SimpleImputer(strategy='median')),
                   ('tree', DecisionTreeClassifier(random_state=SEED))]).fit(Xi_tr, yi_tr)

for name, model in [('always guess "died"', dumb), ('decision tree', tree_i)]:
    pred = model.predict(Xi_te)
    print(f'{name:22s} accuracy {accuracy_score(yi_te, pred):.4f}   F1 (survivors) {f1_score(yi_te, pred):.4f}')

✅ **Expected:**

| Model | Accuracy | F1 (survivors) |
|---|---:|---:|
| always guess "died" | **0.9016** | 0.0000 |
| Decision Tree | 0.8443 | **0.2963** |

**Read that table carefully, because it inverts your instinct.**

By accuracy alone, guessing "died" every time **beats** the decision tree by 5.7 points.
But that winning model **never finds a single survivor** (F1 = 0). It learned nothing; it repeats one answer.

The decision tree, with the lower accuracy, is the only one doing the job we hired it for.

> **Pick a model by accuracy on skewed data and you will pick the worse model every time.**

### 🔵 C.7 — The kind of leakage you cannot see

The leakage in C.3 was loud: 100%. This one is silent.

If you `fit` an imputer or a scaler on the **whole** dataset before splitting into train and test,
the statistics it learns (median, mean, std) contain information from the test set —
so the model has already peeked at part of the exam.

In [ ]:
Xk_tr, Xk_te, yk_tr, yk_te = train_test_split(     # the split this cell compares against
    X[NUM], y, test_size=0.2, random_state=SEED, stratify=y)

median_all   = X['age'].median()
median_train = Xk_tr['age'].median()

print(f'median age from ALL data  : {median_all}')
print(f'median age from TRAIN only: {median_train}')
print(f'mean fare  from ALL data  : {X["fare"].mean():.4f}')
print(f'mean fare  from TRAIN only: {Xk_tr["fare"].mean():.4f}')

✅ **Expected:** median `28.0` vs `28.5` · mean fare `32.2042` vs `31.8198`

**To be straight with you:** on Titanic the difference is tiny and accuracy barely moves.

It becomes serious when the dataset is **small**, or when the transform is stronger
(target encoding, SMOTE). And the real problem is that **you cannot see it happening** —
you find out when the model goes live and the numbers drop.

> **The answer is not to be careful. It is to use a tool that cannot make the mistake** — a `Pipeline`.

### 🔵 C.8 — `Pipeline`, the tool that makes C.7 impossible

A `Pipeline` ties every step together and fits the whole chain on **train only**, automatically.

`ColumnTransformer` sends numeric and text columns down separate routes, because they need different treatment.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

NUM = ['age', 'sibsp', 'parch', 'fare', 'pclass']
CAT = ['sex', 'embarked', 'embark_town', 'alone']

preprocess = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')),
                      ('scale',  StandardScaler())]), NUM),
    ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                      ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), CAT),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)

full = Pipeline([('prep', preprocess),
                 ('model', DecisionTreeClassifier(random_state=SEED))]).fit(X_train, y_train)

print(f'Accuracy: {accuracy_score(y_test, full.predict(X_test)):.4f}')

✅ **Expected:** `Accuracy: 0.8101`

**`handle_unknown='ignore'` matters more than it looks.** If the test set contains a value never seen in
training — a port that only appears once, say — the default behaviour is to crash mid-run.
This option encodes it as all zeros instead.

---
# D · Back to the Model — and Measuring It Properly
🎤 **Walked live:** D.2 · D.5 &nbsp;·&nbsp; 📖 **read on your own:** D.1 · D.3 · D.4 · D.6 · D.7

Part A opened a loop: the model broke because the data was dirty. This part closes it **with numbers**,
and then corrects the most dangerous misunderstanding of the session: accuracy is not the answer.

### 🔵 D.1 — Before and after

In [ ]:
rows = [
    ('Baseline (always "died")', base_acc),
    ('A: tree, 3 raw columns',   acc),
    ('C: tree, full pipeline',   accuracy_score(y_test, full.predict(X_test))),
]
before_after = pd.DataFrame(rows, columns=['Stage', 'Accuracy'])
before_after['vs baseline'] = (before_after['Accuracy'] - base_acc).round(4)
print(before_after.round(4).to_string(index=False))

✅ **Expected:** baseline `0.6145` → three raw columns `0.6480` → full pipeline `0.8101`

Read it two ways:
- **Against the baseline:** the full pipeline is **+19.6 points** better than guessing; the three raw columns
  were only **+3.4** better
- **What part C bought:** `0.8101 − 0.6480 = ` **+16.2 points** — that is the value of preprocessing alone,
  because the model is still the same decision tree

Keep the number **16.2** for part D.2.

### 🔵 D.2 — How much does changing the model help?

Same `preprocess`, only the final estimator changes.

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import roc_auc_score

candidates = {
    'Baseline (majority)': DummyClassifier(strategy='most_frequent'),
    'kNN (k=5)':           KNeighborsClassifier(),
    'Decision Tree':       DecisionTreeClassifier(random_state=SEED),
    'Naive Bayes':         GaussianNB(),
    'Random Forest':       RandomForestClassifier(n_estimators=300, random_state=SEED),
    'Gradient Boosting':   GradientBoostingClassifier(random_state=SEED),
}

results = []
for name, model in candidates.items():
    pipe = Pipeline([('prep', preprocess), ('model', model)]).fit(X_train, y_train)
    pred = pipe.predict(X_test)
    auc  = roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1])
    results.append((name, accuracy_score(y_test, pred), f1_score(y_test, pred), auc))

scores = pd.DataFrame(results, columns=['Model', 'Accuracy', 'F1', 'ROC-AUC'])
print(scores.round(4).to_string(index=False))

✅ **Expected:**

| Model | Accuracy | F1 | ROC-AUC |
|---|---:|---:|---:|
| Baseline (majority) | 0.6145 | 0.0000 | 0.5000 |
| kNN (k=5) | 0.8045 | 0.7328 | 0.8391 |
| Decision Tree | 0.8101 | 0.7463 | 0.7819 |
| Naive Bayes | 0.7765 | 0.7101 | 0.8022 |
| Random Forest | 0.8101 | 0.7344 | 0.8245 |
| Gradient Boosting | 0.7989 | 0.7097 | 0.8211 |

**The most important observation in the session:** the five real models span **0.7765–0.8101** —
**3.4 points** apart. The preprocessing in part C was worth **16.2 points**, nearly **five times** as much.

> Time spent understanding your data pays better than time spent hunting for a fancier model.

**And one more thing to notice:** `Decision Tree` has the highest accuracy (0.8101) but the
**lowest ROC-AUC of the real models** (0.7819), while `kNN` has slightly lower accuracy and the
**highest ROC-AUC** (0.8391).

They measure different things. Accuracy asks how many predictions are right at a 0.5 threshold.
ROC-AUC asks how well the model **ranks** risk, across every threshold.
If the job is "send the 20 highest-risk people for further screening", ROC-AUC is the relevant one.

### 🔵 D.3 — Overfitting, as a picture

`max_depth` is the complexity dial on a tree. Sweep it and plot train and test scores together.

In [ ]:
depths = [1, 2, 3, 4, 5, 7, 10, 15, 20]
train_scores, test_scores = [], []

for d in depths:
    p = Pipeline([('prep', preprocess),
                  ('model', DecisionTreeClassifier(max_depth=d, random_state=SEED))]).fit(X_train, y_train)
    train_scores.append(p.score(X_train, y_train))
    test_scores.append(p.score(X_test, y_test))

plt.figure(figsize=(7, 4))
plt.plot(depths, train_scores, 'o-', label='train', color='tab:blue')
plt.plot(depths, test_scores, 's-', label='test', color='tab:orange')
plt.fill_between(depths, test_scores, train_scores, color='tab:red', alpha=0.08)
plt.text(10.4, 0.885, 'the gap is what the model memorised', color='tab:red', fontsize=9)
plt.xlabel('max_depth'); plt.ylabel('accuracy')
plt.title('The tree keeps learning the training set - and stops helping on new data')
plt.legend(); plt.ylim(0.6, 1.02); plt.tight_layout(); plt.show()

print(pd.DataFrame({'max_depth': depths, 'train': np.round(train_scores, 4),
                    'test': np.round(test_scores, 4)}).to_string(index=False))

✅ **Expected:** train climbs from `0.7893` to `0.9817`, while test wanders around `0.76–0.82` and goes nowhere

**This is what overfitting looks like** — the red area between the lines is what the model memorised
and cannot reuse on anyone new.

This chart is the **bias–variance tradeoff** your lectures cover, drawn from real numbers:

| End of the curve | What is happening | The name for it |
|---|---|---|
| `max_depth=1`, both scores low and close | the model is too simple to capture the pattern — it is wrong in the same way every time | **high bias** (underfitting) |
| `max_depth=20`, train 0.98 and test 0.81 | the model has memorised this particular sample — a different sample would give a different model | **high variance** (overfitting) |

The useful depth is where the two lines are closest without both being low, and the red area is the
variance you are paying for.

**Notice something else:** the test line has no clear peak. `max_depth=15` gives `0.8212`,
but `max_depth=10` gives `0.7989`, which is worse than `max_depth=7`.
That wobble is not a signal from the data — it is **noise from splitting the test set once.**
Picking `max_depth` straight off this chart means picking whichever value got lucky,
which is exactly why the next section exists.

### 🔵 D.4 — One split is not enough

Every number so far comes from **one** train/test split, with `random_state=42`.
Change that number and the results change. So which one do you report?

**Cross-validation** splits several times and reports both the average and the spread.

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
rf = Pipeline([('prep', preprocess),
               ('model', RandomForestClassifier(n_estimators=300, random_state=SEED))])

folds = cross_val_score(rf, X, y, cv=cv, scoring='f1')
print('F1 per fold:', np.round(folds, 4))
print(f'mean {folds.mean():.4f}  ±{folds.std():.4f}')

✅ **Expected:** `[0.7852 0.7465 0.7015 0.7669 0.7669]` · mean `0.7534` ±`0.0287`

Lowest and highest fold are **8 points** apart, on the same data with the same model.

> **How to report a result:** write `0.75 ± 0.03`, not `0.7852`.
> Reporting a single number from a single split is reporting your luckiest value.

### 🔵 D.5 — How is the model wrong?

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

rf_fitted = rf.fit(X_train, y_train)
pred = rf_fitted.predict(X_test)

cm = confusion_matrix(y_test, pred)
print(cm)
print()
print(classification_report(y_test, pred, target_names=['died', 'survived'], digits=3))

✅ **Expected:** the matrix `[[98 12] [22 47]]`, and recall for `survived` = `0.681`

**Reading the matrix:**

| | predicted: died | predicted: survived |
|---|---:|---:|
| **actually died** | 98 ✅ | 12 ❌ *false positive* |
| **actually survived** | **22 ❌ *false negative*** | 47 ✅ |

The model **misses 22 of the 69 survivors** — recall of 68%, while accuracy reads 81%.
The accuracy figure never mentioned this.

### 🔵 D.6 — The question to answer before choosing a metric

> **With more than two classes, use `average='macro'`.** It averages the classes evenly, so a class
> holding 18% of the rows counts as much as one holding 63% — and a model that quietly ignores the
> small class cannot hide behind a good-looking score.

Suppose this were **screening patients for heart-attack risk**, where 1 = at risk.

- **False negative** = telling a sick patient they are fine → they go home untreated
- **False positive** = telling a healthy patient they are at risk → extra tests, cost and worry, but they are safe

The two errors cost wildly different amounts. **Screening therefore optimises recall**, and accepts
a lot of false positives to get it.

> **The order is: understand the problem → choose the metric → then tune the model.**
> Not: pick whichever metric makes the number look best.

### 🎨 D.7 — Chart polish, step 1

Every session ends with ten minutes spent on a chart you just made yourself. **Today's two techniques:**

1. **A chart title should be the conclusion, not the column name.** `"Model accuracy"` tells the reader nothing.
   `"Every model lands within 3 points of the others"` tells them the finding.
2. **A y-axis that does not start at zero magnifies small differences** — the most common way to lie with a chart.

In [ ]:
plot_df = scores[scores.Model != 'Baseline (majority)']

fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))

# left: the misleading version
ax[0].bar(range(len(plot_df)), plot_df.Accuracy, color='tab:blue')
ax[0].set_ylim(0.77, 0.815)
ax[0].set_xticks(range(len(plot_df)))
ax[0].set_xticklabels(plot_df.Model, rotation=20, ha='right', fontsize=8)
ax[0].set_title('Model accuracy')                       # column name as a title
ax[0].set_ylabel('accuracy')

# right: honest axis + a title that states the finding
colors = ['tab:grey'] * len(plot_df)
colors[int(np.argmax(plot_df.Accuracy.values))] = 'tab:blue'
ax[1].bar(range(len(plot_df)), plot_df.Accuracy, color=colors)
ax[1].axhline(base_acc, color='tab:red', ls='--', lw=1)
ax[1].text(-0.4, base_acc + 0.012, f'baseline {base_acc:.2f}', color='tab:red', fontsize=9)
ax[1].set_ylim(0, 1)
ax[1].set_xticks(range(len(plot_df)))
ax[1].set_xticklabels(plot_df.Model, rotation=20, ha='right', fontsize=8)
ax[1].set_title('Every model lands within 3 points - the data mattered more than the model')
ax[1].set_ylabel('accuracy')

plt.tight_layout(); plt.show()

✅ **Expected:** two panels built from **the same numbers** that tell different stories

The left one makes the Decision Tree look like a clear winner. The right one shows every model bunched
together, with a baseline to measure against.

**The right one is the honest chart:** its axis starts at zero, everything is grey except the bar being
pointed at, and the title states the finding instead of naming the chart.

✅ **Expected:** `usa 0.626` · `japan 0.198` · `europe 0.176`

**1 · Is it balanced?** No. American cars are **62.6%** of the rows, and Europe is **17.6%** —
a factor of 3.6 between the largest and smallest class.

**2 · Which error costs more?** For "where was this car built" the two mistakes cost the same, so
nothing pushes us towards recall or precision. **That is itself an answer**, and it is worth saying out
loud rather than assuming.

**3 · The metric.** With 62.6% in one class, a model that says *usa* every time already scores 0.626.
So accuracy alone would flatter it. **Report macro-F1 next to accuracy** — macro averages the three
classes evenly, so europe counts as much as usa, and a model that ignores the small classes cannot hide.

> **Compare with the titanic case in C.6.** There the two errors were *not* equal — missing a survivor
> is not the same as a false alarm — so the reasoning ended somewhere else. **The metric follows from
> the problem, not from a rule you memorise.**

---
---
# 📋 Part 2 — Pick Your Topic
### Assignment 1 · one of these 14 · the demo is over, this takes ~8 minutes

**Groups of three or four. One topic per group, first come first served, no two groups on the same one.**

**This list is session 1's.** Every session has its own, 12–15 topics each, and you pick again every time.
From session 2 the list comes out a week ahead and **you arrive with the data already loaded** — session 1
is the only one where you pick in the room, which is why this hour is the tightest of the six.

Take the topic you pick and go the whole way:

```
real data -> EDA -> preprocessing (Pipeline) -> baseline -> model -> a metric that fits -> limitations
```

📄 **[How it is marked, how the questions work, what to hand in](https://classes.incortx.com/DataAnalytics/session-00/)** — read that before you start.
The short version: you need **a baseline to beat**, a **`Pipeline` with no leakage**, and **a metric that
fits the problem**. A model that loses to its own baseline still scores full marks if you explain why.

**The 14 topics.** Difficulty: 🟢 the data is ready to use · 🟡 it needs real cleaning or merging first. Every dataset here already exists and is free to download — none of them ask you to collect or scrape data yourself, so a week is enough for any of them.

The **trap** column is not a general warning. It is the specific thing that will bite you in that dataset,
and it is where the code questions will come from.

**Health & the body**

| # | Topic — the question to answer | Data | The trap you will hit | |
|:--:|---|---|---|:--:|
| **1** | Screen for diabetes risk from basic measurements | `pd.read_csv` on one URL — `raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv` (no header row, so pass `names=[...]`) | Glucose, blood pressure, skin thickness, insulin and BMI all contain **zeros**, and a living person cannot have a BMI of 0. They are missing values written as 0, so `isna()` reports nothing and `SimpleImputer` cannot see them either. Turn them into `np.nan` yourself first. | 🟢 |
| **2** | Is this tumour benign or malignant? | `sklearn.datasets.load_breast_cancer(as_frame=True)` — one call, nothing to download | The 30 columns are really **10 measurements seen three ways** (mean, error, worst). `mean radius` and `worst radius` correlate at 0.97, so the same evidence is counted three times and any importance ranking you read off is misleading. | 🟢 |
| **3** | Which penguin species is this? | `seaborn.load_dataset('penguins')` — one call | Missing values in five columns, including 11 in `sex`, so you must decide row-drop against impute and say why. And the three species are **44 / 36 / 20 percent**, so the smallest one is easy for a model to quietly ignore. | 🟢 |
| **4** | Will this donor come back to give blood again? | `sklearn.datasets.fetch_openml('blood-transfusion-service-center', version=1, as_frame=True)` | Only **748 rows and 76.2% in one class**. Small and skewed together means a single train/test split will bounce around depending on the seed — this is the topic where cross-validation stops being optional. | 🟡 |

**Food, drink & nature**

| # | Topic — the question to answer | Data | The trap you will hit | |
|:--:|---|---|---|:--:|
| **5** | Is this mushroom safe to eat? | `sklearn.datasets.fetch_openml('mushroom', version=1, as_frame=True)` | You will hit **100.00% accuracy** almost immediately, and that is the whole lesson. **Eight of the nine `odor` values decide the answer on their own**, so the model has found one column, not learned biology. Your job is to show that and say what it means. | 🟢 |
| **6** | Which of three vineyards made this wine? | `sklearn.datasets.load_wine(as_frame=True)` — one call | `proline` spans about 1402 and `nonflavanoid_phenols` spans 0.53 — a **2,645x** difference. A tree does not care; anything measuring distance does. This is the clearest place to show the same data scored twice, with and without scaling. | 🟢 |
| **7** | Which of the three iris species is this? | `sklearn.datasets.load_iris(as_frame=True)` — the classic first dataset | The easiest dataset here, and it still **stops around 0.93**. Two of the three species overlap, so no amount of model choice reaches 1.0. The interesting question is not the score, it is showing *which* two get confused and why. | 🟢 |
| **8** | What cut grade was this diamond given? | `seaborn.load_dataset('diamonds')` — one call, 53,940 rows | `cut` is **ordinal** — Fair < Good < Very Good < Premium < Ideal — but sorted alphabetically it becomes Fair, Good, Ideal, Premium, Very Good. Encode it the default way and you have taught the model an order that does not exist. Also 40% are Ideal and 3% are Fair. | 🟡 |

**Money & risk**

| # | Topic — the question to answer | Data | The trap you will hit | |
|:--:|---|---|---|:--:|
| **9** | Should this loan application be approved? | `sklearn.datasets.fetch_openml('credit-g', version=1, as_frame=True)` | 70% of applications are good, so "approve everything" already scores 0.70. Worse, **the two mistakes do not cost the same** — approving a bad loan loses the money, refusing a good one loses a customer. This is the topic where you must argue for recall or precision rather than accuracy. | 🟡 |
| **10** | Who earns above the census income threshold? | `sklearn.datasets.fetch_openml('adult', version=1, as_frame=True)` | **76.1% of people are below the threshold**, so a model that always says "below" reports 76% accuracy while finding nobody. There is also an `fnlwgt` column that is a survey sampling weight describing how the census was drawn, not a fact about the person. | 🟡 |
| **11** | Did this taxi passenger pay by cash or by card? | `seaborn.load_dataset('taxis')` — one call, 6,433 rides | **`tip` is exactly 0 for 100% of cash rides** and for only 9.9% of card rides, because cash tips are never recorded. Keep that column and you get a near-perfect score from a column that *is* the answer. This is the leakage demo, on a dataset where you have to spot it yourself. | 🟡 |
| **12** | Is this banknote genuine or forged? | `sklearn.datasets.fetch_openml('banknote-authentication', version=1, as_frame=True)` | Four features and the two classes separate almost perfectly, so everything you try scores near 1.0. That makes the report the hard part: **what have you actually learned when the problem was easy?** Compare against the baseline and say honestly how much the model added. | 🟢 |

**Signals & text**

| # | Topic — the question to answer | Data | The trap you will hit | |
|:--:|---|---|---|:--:|
| **13** | Which phoneme class is this sound? | `sklearn.datasets.fetch_openml('phoneme', version=1, as_frame=True)` | Five numeric features and nothing else — no names, no units, no meaning you can look up. **70.7% sit in one class.** With no domain knowledge available, everything has to come from the numbers, which makes the metric argument unavoidable. | 🟡 |
| **14** | Was this restaurant bill a lunch or a dinner? | `seaborn.load_dataset('tips')` — one call, 244 rows | **244 rows is the whole dataset**, and 72.1% are dinner. A 20% test split leaves about 49 rows, so a single accuracy figure moves by several points if you change the random seed. Show that happening rather than reporting one number as if it were solid. | 🟡 |

In [ ]:
# ── Record your group's choice ─────────────────────────────────────────────
TOPIC_ID = None      # <- put your group's topic number here, then run this cell

TOPIC_TRAPS = {
     1: ('Screen for diabetes risk from basic measurements',
        'zeros in glucose, BMI and blood pressure are missing values in disguise'),
     2: ('Is this tumour benign or malignant?',
        '30 columns are 10 measurements x 3 views, correlated at 0.97'),
     3: ('Which penguin species is this?',
        'NaNs in five columns, and Chinstrap is only 20% of the rows'),
     4: ('Will this donor come back to give blood again?',
        '748 rows and 76% one class, so one split is not enough'),
     5: ('Is this mushroom safe to eat?',
        'you get 100% accuracy because odor alone decides it'),
     6: ('Which of three vineyards made this wine?',
        'proline and nonflavanoid_phenols are 2,645x apart in scale'),
     7: ('Which of the three iris species is this?',
        'the ceiling is about 0.93 because two species overlap'),
     8: ('What cut grade was this diamond given?',
        'cut is ordinal but sorts alphabetically into the wrong order'),
     9: ('Should this loan application be approved?',
        '70% are good, and the two kinds of mistake cost very different amounts'),
    10: ('Who earns above the census income threshold?',
        'always saying "below" scores 76%, and fnlwgt is a survey weight'),
    11: ('Did this taxi passenger pay by cash or by card?',
        'tip is 0 for every single cash ride, so it leaks the answer'),
    12: ('Is this banknote genuine or forged?',
        'almost perfectly separable, so the score tells you very little'),
    13: ('Which phoneme class is this sound?',
        '70.7% in one class, and five features with no meaning attached'),
    14: ('Was this restaurant bill a lunch or a dinner?',
        '244 rows total, so a single split is not a stable answer'),
}

if TOPIC_ID in TOPIC_TRAPS:
    title, trap = TOPIC_TRAPS[TOPIC_ID]
    print(f'Topic {TOPIC_ID}: {title}')
    print(f'Watch out for : {trap}')
else:
    print('Set TOPIC_ID to your group number (1-14) and run this cell again.')

✅ **Expected:** your topic and its trap printed back at you. Write the trap somewhere you will see it again —
it is the first thing you should check when your results look strange.

---
---
# 🟠 Part 3 — Your Hour · 60 Minutes, Your Own Data

**Everything above was the demo.** From here it is your group's work, and it is what gets marked.

This section does not depend on a single cell above it. Run it from the top of this section and it works.

### The steps, and the clock

| Minutes | Step | What has to exist when you are done |
|:--:|---|---|
| 0–10 | **0 · Load your data** | `shape` · `head()` · one sentence saying what one row is |
| 10–22 | **1 · EDA → one insight** | a chart, and a sentence stating what you *found* — not what the chart is |
| 22–35 | **2 · Prepare the data** | what was wrong, what you did, **and why that choice** — inside a `Pipeline` |
| 35–45 | **3 · Metric + baseline** | the metric's name **with a reason**, and the baseline measured with that same metric |
| 45–55 | **4 · Today's technique** *(if you get there)* | a score, printed next to the baseline |
| 55–60 | **5 · Write up + get ready** | one sentence, one limitation, notebook scrolled to where you will start talking |

**Steps 1, 2 and 3 are what you present and what is marked. Step 4 is a bonus.** A group that never
trains a model but has a real insight, a justified metric and a baseline scores full marks.

**It does not have to be finished or pretty.** If a step defeats you, write what stopped you — that
scores better than deleting it. At the end your group puts this on screen for six minutes.

> **Only session 1 loads data during the hour.** From session 2 the topic bank comes out a week early
> and you arrive with `df` already loaded — loading eats 8–12 minutes and carries no marks.

### 🟠 The submission header

**Everyone in the group presents, every assignment.** The four minutes in the room are shared by **two**
speakers; the rest of the group presents the same work in the **homework video**. Fill in who does which
before you start — deciding it at 59 minutes is how groups end up with one person doing all the talking.

| Group of | In the room (4 min) | In the video (5 min) | Over six sessions |
|:--:|:--:|:--:|---|
| 3 | 2 people | 1 person | 4 turns in the room, 2 in the video, each |
| 4 | 2 people | 2 people | 3 and 3 each — and every pair works together once |

**Every session has both stages**, so the rotation comes out even. Nobody can spend the term in the
video, which is the easier of the two.

In [ ]:
# ── SUBMISSION HEADER — fill this in first ─────────────────────────────────
GROUP     = ''            # your group letter: 'A' .. 'J'
MEMBERS   = ['', '', '']  # everyone in the group - keep this order all term
TOPIC_ID  = None          # the topic number your group claimed

IN_CLASS  = ['', '']      # the TWO presenting in the room today
IN_CLIP   = ['']          # the rest, presenting in the homework video

# ── check: everyone presents somewhere ─────────────────────────────────────
_all   = [m.strip() for m in MEMBERS  if m.strip()]
_room  = [m.strip() for m in IN_CLASS if m.strip()]
_clip  = [m.strip() for m in IN_CLIP  if m.strip()]

print(f'Group {GROUP or "?"} | topic {TOPIC_ID} | {len(_all)} members')
print(f'  in the room : {", ".join(_room) or "-- nobody --"}')
print(f'  in the video: {", ".join(_clip) or "-- nobody --"}')

missing = [m for m in _all if m not in _room + _clip]
twice   = [m for m in _room if m in _clip]
if not GROUP or not _all or TOPIC_ID is None:
    print('\n[ ] header not filled in yet')
elif missing:
    print(f'\n[!] not presenting anywhere: {", ".join(missing)} - everyone has to present')
elif twice:
    print(f'\n[!] listed twice: {", ".join(twice)} - pick one or the other')
elif len(_room) != 2:
    print(f'\n[!] {len(_room)} in the room, should be 2')
else:
    print('\n[ok] everyone presents')

### 🟠 Setup for this section

Its own imports, so this half runs whatever happened above.

In [ ]:
# ── Everything the twenty topics need. Run this once; nothing above is required.
import warnings; warnings.filterwarnings('ignore')

# every session-1 topic is a library call or one read_csv(url) — no zips this time
import io as _io, zipfile, urllib.request      # here for later sessions

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# the loaders some topics use instead of a URL
from sklearn.datasets import fetch_openml, load_breast_cancer, load_wine, load_iris

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             classification_report, mean_absolute_error,
                             mean_squared_error, r2_score)

SEED = 42
np.random.seed(SEED)
pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 30)
plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.grid'] = True; plt.rcParams['grid.alpha'] = 0.3
print('ready')

**Two helpers come with the notebook: `eda()` and `baseline()`.** Run the cell below once and both are
available for the rest of the hour.

**Neither one does the thinking.** `eda()` prints the six views you should always look at; `baseline()`
prints the number your model has to beat. **Reading them is the part that carries marks** — and it is
the part nobody can do for you.

In [ ]:
def eda(df, target=None, n=6):
    """Print the six things you should always look at first. Reading them is your job.

        eda(df)                    # no target column yet
        eda(df, target='outcome')  # adds class balance + correlation with the target
    """
    import pandas as _pd
    line = '\u2500' * 62

    print(line); print(f'1. SHAPE      {df.shape[0]:,} rows x {df.shape[1]} columns')
    print(line); print('2. ONE ROW    what does a single row actually represent?')
    print(df.head(3).to_string())

    print(line); print('3. TYPES      a number stored as text will not go into a model')
    info = _pd.DataFrame({'dtype': df.dtypes.astype(str),
                          'non_null': df.notna().sum(),
                          'distinct': df.nunique()})
    print(info.to_string())

    print(line); print('4. MISSING    how much, and in which columns')
    miss = df.isna().sum()
    miss = miss[miss > 0].sort_values(ascending=False)
    if len(miss) == 0:
        print('   isna() finds none  --  but a 0 or a -200 can be a missing value in disguise.')
        print('   Check step 5 for impossible values before you believe this.')
    else:
        print(_pd.DataFrame({'missing': miss, 'percent': (miss / len(df) * 100).round(1)}).to_string())

    print(line); print('5. RANGES     look for a min or max that cannot be real')
    num = df.select_dtypes('number')
    print(num.describe().T[['min', '25%', '50%', '75%', 'max']].to_string() if len(num.columns)
          else '   no numeric columns')

    if target is not None and target in df.columns:
        print(line); print(f'6. TARGET     {target!r}')
        y = df[target]
        if y.dtype.kind == 'f' or y.nunique() > 20:
            print(y.describe().to_string())
            corr = num.corr(numeric_only=True)[target].drop(target).sort_values(key=abs, ascending=False)
            print(f'\n   strongest correlations with {target}:')
            print(corr.head(n).round(4).to_string())
        else:
            print(y.value_counts(normalize=True).round(4).to_string())
            print(f'\n   most common class is {y.value_counts(normalize=True).max():.1%} of rows')
    print(line)
    print('Now write ONE sentence about something you did not know 60 seconds ago.')

def baseline(y_train, y_test, kind='auto'):
    """Print the score of the dumbest possible model for this target.

    You do not have to write a baseline. You do have to read it and say what it
    means -- that is the part that carries marks.

        baseline(y_tr, y_te)            # works out classification vs regression
        baseline(y_tr, y_te, 'clf')     # force it
        baseline(y_tr, y_te, 'reg')
    """
    import numpy as _np, pandas as _pd
    from sklearn.dummy import DummyClassifier, DummyRegressor
    from sklearn.metrics import (accuracy_score, f1_score, mean_absolute_error,
                                 mean_squared_error, r2_score)

    ytr, yte = _pd.Series(y_train), _pd.Series(y_test)
    if kind == 'auto':
        numeric = _pd.api.types.is_numeric_dtype(ytr)
        kind = 'reg' if (numeric and (ytr.dtype.kind == 'f' or ytr.nunique() > 20)) else 'clf'
        looks = 'a number -> regression' if kind == 'reg' else 'a label -> classification'
        print(f'[auto] your target looks like {looks}'
              f'  ({ytr.nunique()} distinct values, dtype {ytr.dtype})')
        print("       wrong guess? pass kind='clf' or kind='reg'")

    Xtr = _np.zeros((len(ytr), 1))          # a baseline ignores the features on purpose
    Xte = _np.zeros((len(yte), 1))

    if kind == 'clf':
        m = DummyClassifier(strategy='most_frequent').fit(Xtr, ytr)
        p = m.predict(Xte)
        top = m.classes_[0]
        top = top.item() if hasattr(top, 'item') else top
        acc = accuracy_score(yte, p)
        f1 = f1_score(yte, p, average='binary' if yte.nunique() == 2 else 'macro',
                      zero_division=0)
        print(f'baseline = always predict the most common class ({top!r})')
        print(f'  accuracy {acc:.4f}')
        print(f'  F1       {f1:.4f}   <- same model. If these two disagree, accuracy is the wrong metric')
        return {'accuracy': acc, 'f1': f1}

    m = DummyRegressor(strategy='median').fit(Xtr, ytr)
    p = m.predict(Xte)
    mae = mean_absolute_error(yte, p)
    rmse = mean_squared_error(yte, p) ** 0.5
    r2 = r2_score(yte, p)
    print(f'baseline = always predict the training median ({_np.median(ytr):.4f})')
    print(f'  MAE  {mae:.4f}')
    print(f'  RMSE {rmse:.4f}')
    print(f'  R2   {r2:.4f}   <- a baseline R2 at or just below zero is correct, not a bug')
    return {'mae': mae, 'rmse': rmse, 'r2': r2}

print('eda() and baseline() ready')

---
## 📘 Guides — read this if you have never done this before

Five short guides. **You do not need to read them end to end** — jump to the one you are stuck on.

### 📘 1 · The EDA checklist

`eda(df, target='your_target_column')` prints all six of these at once. Here is what each one is for,
and **what "a problem" looks like** — because seeing the number is easy and knowing it is wrong is not.

| # | What it shows | You are looking for |
|:--:|---|---|
| 1 | **shape** | 200 rows is a different job from 200,000 |
| 2 | **one row** | if you cannot say what one row *is*, nothing later will make sense |
| 3 | **types + distinct** | a number stored as text (`dtype object`) · a column with 1 distinct value (useless) · an ID column with as many distinct values as rows |
| 4 | **missing** | any column above ~40% · and remember `isna()` **cannot see a 0 or a −200 that means "missing"** |
| 5 | **ranges** | **a min or max that cannot be real** — a BMI of 0, an age of 200, a max 100× the 75th percentile |
| 6 | **target** | one class over 70% (accuracy is now the wrong metric) · **a correlation of 1.00 with the target, which is leakage** |

**Row 5 and row 6 are where the trap in your topic usually shows up.** Look there first.

Run the individual commands yourself if you want to dig further:

```python
df.shape
df.head()
df.info()
df.describe()
df.isna().sum()
df['target'].value_counts(normalize=True)
df.corr(numeric_only=True)['target'].sort_values(key=abs, ascending=False)
```

### 📘 2 · Which chart answers which question

**Do not open a chart gallery and pick a pretty one.** Start from the question.

| The question in your head | The chart | One line of code |
|---|---|---|
| is this column skewed? | histogram | `df['col'].hist(bins=30)` |
| are my classes balanced? | count plot | `sns.countplot(data=df, x='target')` |
| are these two related? | scatter | `sns.scatterplot(data=df, x='a', y='b')` |
| does this number differ by group? | box plot | `sns.boxplot(data=df, x='group', y='num')` |
| what is related to what? | heatmap | `sns.heatmap(df.corr(numeric_only=True), annot=True, fmt='.2f')` |
| where is the data missing? | missing map | `sns.heatmap(df.isna(), cbar=False)` |

Always add `plt.title(...)` and `plt.tight_layout()`, then `plt.show()`. **A chart with no title is a
chart nobody can mark.**

### 📘 3 · Turning a chart into an insight

An insight is not a description. Fill in this sentence and you have one:

> **"We saw ___ , which means ___ , so we ___ ."**
> *(what is on the chart)* · *(what it means for our question)* · *(what we did about it)*

| Not an insight | An insight |
|---|---|
| "This is a histogram of BMI." | "12 rows have a BMI of 0, which is not a measurement, so we turned those into NaN before imputing." |
| "The classes are 65/35." | "Only 35% are positive, so accuracy would flatter a model that never predicts positive — we report F1." |
| "These two columns correlate." | "`casual + registered` equals the target exactly, so keeping them is leakage — we dropped both." |

**The right-hand column changes what you do next. That is the whole test.**

### 📘 4 · The six errors you will actually hit

| The message | What it means | The fix |
|---|---|---|
| `KeyError: 'xxx'` | that column name does not exist | `print(df.columns.tolist())` and copy the real name |
| `ValueError: could not convert string to float` | a text column went into a model | one-hot it, or drop it for now |
| `ValueError: Input contains NaN` | missing values reached the model | put `SimpleImputer` in the `Pipeline` **before** the model |
| `Found input variables with inconsistent numbers of samples` | `X` and `y` are different lengths | you filtered one and not the other — rebuild both from the same `df` |
| `NameError: name 'df' is not defined` | you skipped a cell | Runtime → *Run before* |
| `SettingWithCopyWarning` | you sliced then assigned | use `df.loc[rows, 'col'] = ...`, or `.copy()` when you slice |

**Read the LAST line of the error first.** Everything above it is where Python has been, not what went wrong.

### 📘 5 · When you are stuck

**Three minutes on one error, then move on.** The clock does not stop and step 4 is not what is marked.

```python
# TODO: ran out of time here.
# We wanted to one-hot `embarked` but kept getting "Input contains NaN".
# We think the imputer needs to come before the encoder in the Pipeline.
```

**A cell like that scores. A deleted cell does not.** Saying where you got stuck and what you think is
wrong is evidence that you understood the problem — which is what the marks are for.

**Nothing in session 1 needs installing.** All fourteen topics run on what Colab already has —
that is why they were chosen. Later sessions will need the occasional `!pip install`.

### 🟠 Getting your file in

Section B covered the four shapes. Here they are again in one place, because you need them now and
scrolling back costs you minutes you do not have.

```python
# 1 · a library call  — most topics
df = load_wine(as_frame=True).frame
df = sns.load_dataset('penguins')
df = fetch_openml('adult', version=1, as_frame=True).frame   # first call ~10s

# 2 · a CSV at a URL  — topic 1
df = pd.read_csv(URL, header=None, names=[...])   # header=None if there is no header row

# 3 · a zip at a URL  — 11 topics
with urllib.request.urlopen(URL) as r:
    z = zipfile.ZipFile(_io.BytesIO(r.read()))
print(z.namelist())                                # always look first
df = pd.read_csv(z.open('the_file.csv'), sep=';')  # sep=';' where the topic says so

# 4 · a zip inside a zip — topics 9, 18
inner = zipfile.ZipFile(_io.BytesIO(z.read('inner.zip')))
df = pd.read_csv(inner.open('the_file.csv'), sep=';')
```

### 🟠 Step 0 — Load your data · *0–10 min*

Your topic's entry names the file and the separator. Section B showed the four shapes; use the one
that matches. **Print `namelist()` first if it is a zip**, and remember `header=None` if there is no
header row.

**Then run `eda(df, target='...')` immediately** — it takes a fraction of a second and prints all six
views at once, so you start step 1 already knowing where to look. (Guide 1 above says what each view is for.)

Finish with one sentence: **what is one row of this data?**

In [ ]:
# TODO: load your group's data into `df`, then run eda() on it

df = None
# eda(df, target='...')

✅ **What topic 1's `eda()` hands you in under a second**

- **view 4 says `isna()` finds none** — and view 5 immediately contradicts it: `glucose`, `bp`, `skin`,
  `insulin` and `bmi` all have a **min of 0**, which is impossible for a living person. The missing
  values are real, they are just written as zeros. **`isna()` cannot see them and neither can `SimpleImputer`.**
- **view 6 says the majority class is 65.1%** — so accuracy above 65% means nothing on its own.

Those two lines are your step 1 insight, found before you drew a single chart.

**One row of this data is:** *(write it here)*

### 🟠 Step 1 — EDA → one insight · *10–22 min*

Not a pretty chart. A chart that tells you something you did not know a minute ago — where values are
missing, how skewed the target is, which class dominates, where the outliers sit.

**The chart is not the deliverable. The sentence under it is.** Compare:

| Not an insight | An insight |
|---|---|
| "This is a histogram of BMI." | "Twelve rows have a BMI of 0, which is not a measurement — it is a missing value in disguise." |
| "The classes are 65/35." | "Only 35% are positive, so accuracy will look good even if the model never predicts positive." |
| "These two columns are correlated." | "`casual + registered` adds up to the target exactly, so keeping them is leakage." |

The right-hand column changes what you do next. The left-hand column does not. **This is where the trap
in your topic normally shows up**, and it is what criterion C marks.

In [ ]:
# TODO: one chart that shows a problem in your data

**What I found:** *(one sentence — a finding, not a description of the chart)*

**What this changes about what I do next:** *(write it here)*

### 🟠 Step 2 — Prepare the data · *22–35 min*

Everything step 1 told you was wrong, you now fix — **and you write down why you fixed it that way.**
The fix is worth half a mark. The reason is worth the other half.

The four decisions almost every dataset forces on you:

| Decision | The question you have to answer |
|---|---|
| **Missing values** | drop the rows, drop the column, or impute? A column that is 49% missing is a different decision from one that is 1% missing |
| **Impute with what** | mean or median? median if the column is skewed — and you can see that from step 1's chart |
| **Categorical columns** | one-hot? how many categories will that create? |
| **Scale** | trees do not care. Distances, and anything with a gradient, care a lot |

**Two hard rules, and breaking either one costs you criterion B:**

1. **Put it in a `Pipeline`, do not edit `df` in place.** A `Pipeline` learns its fill values and scaling
   from the training rows only, which is the whole point.
2. **`train_test_split` comes before any `fit`.** If you impute on the full table first, the test rows
   have already leaked into the training set — and your score is fiction.

> **`SimpleImputer` cannot see a zero that means "missing".** Turn those into `np.nan` yourself first —
> that is a decision only a human who looked at the data can make, and it is exactly what step 1 is for.

In [ ]:
# TODO: turn impossible values into NaN, split X/y, then build the Pipeline

**What I fixed, and why I chose that fix:** *(write it here — one line per decision)*

### 🟠 Step 3 — Metric + baseline · *35–45 min*

Two things that only mean something together: **the number you will report, and the number it has to beat.**

#### First: which metric, and why *this* metric for *this* problem

"We used accuracy" is not an answer. **"We used F1 because only 35% of our rows are positive, so a model
that never predicts positive still scores 65% accuracy"** is an answer, and it is worth half of criterion B.

| What step 1 showed you | The metric that follows |
|---|---|
| classes are roughly balanced | **accuracy** is fine, and is the easiest to explain |
| one class is 70%+ of the rows | **F1** or **balanced accuracy** — accuracy will flatter a useless model |
| missing the positive class is expensive (disease, fraud) | **recall** first, and say what you are trading away |
| a number, and big errors are much worse than small ones | **RMSE** — it squares the errors, so outliers dominate |
| a number, and all errors count the same | **MAE** — it reads in the units of the target |

**Print both accuracy and F1 if you are unsure.** Two numbers that disagree is itself the finding.

#### Then: the baseline, measured with that same metric

**The code is written for you.** One call, and it works out classification vs regression on its own:

```python
baseline(y_train, y_test)              # or kind='clf' / kind='reg' to force it
```

It prints the score of a model that ignores every feature — always the most common class, or always the
training median. **That number is the floor.** Your job is the sentence next to it, not the code.

**Which one is yours?** Look at the column you are predicting.

| Your target column | Baseline | The one line |
|---|---|---|
| a **label** — survived / diabetic / spam | guess the most common class every time | `DummyClassifier(strategy='most_frequent')` |
| a **number** — price, mpg, count | guess the median every time | `DummyRegressor(strategy='median')` |

Both are already imported. Both are fitted and scored exactly like a real model:

```python
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED)

base = DummyClassifier(strategy='most_frequent').fit(X_tr, y_tr)   # or DummyRegressor(strategy='median')
base_score = accuracy_score(y_te, base.predict(X_te))              # or mean_absolute_error(...)
print(f'baseline: {base_score:.4f}')
```

**Two rules that still belong to you:**

1. **Call it with the same `y_train` / `y_test` you gave your model.** A baseline from a different split
   compares to nothing. `baseline()` cannot check this for you.
2. **From here on, the baseline goes next to every number you print.** Not in your head. In the cell.

**Then read what came back.** This is the marked part:

- A classification baseline of **0.95** means your data is 95:5 imbalanced, and accuracy has just become
  the wrong metric. Use F1, and say that is why.
- **Accuracy and F1 from the same baseline can disagree completely** — that is the single most useful
  thing this function tells you.
- A regression **R² at or just below zero is correct, not a bug.** R² = 0 is predicting the mean of the
  *test* set, which you never get to see; the median of *train* lands slightly under. Demo E.2 got `-0.0502`.

In [ ]:
# TODO: split, name your metric and why, then call baseline()

✅ **Expected on topic 1:** baseline accuracy `0.6494` · baseline **F1 `0.0000`**

**Look at those two numbers.** Identical model. One says it gets two thirds of the answers right, the
other says it is worthless. Both are true: it labels everyone "not diabetic", so it is right 64.94% of
the time and catches zero actual cases.

**This is why the metric needs a reason.** Pick accuracy here and every model you try will look fine.

**Our metric is ___ because ___** *(fill this in — it is half of criterion B)*

### 🟠 Step 4 — Today's technique · *45–55 min* — **if you get there**

**This step is a bonus, not a requirement.** If steps 1–3 took the whole hour, you still score full marks.
Write "ran out of time at step 4" and move to step 5 — that is a better use of the last five minutes
than a model nobody can explain.

Your `prep` from step 2 is already built. Drop a model on the end of it and score it:

```python
Pipeline([('prep', prep), ('model', DecisionTreeClassifier(max_depth=4, random_state=SEED))])
```

Then print your score **next to the baseline**, never alone.

Before you run it, ask the question from the demo: **is any column in here the answer in disguise?**

In [ ]:
# TODO: put a model on the end of `prep`, and score it next to the baseline

✅ **Expected on topic 1:**

| Model | Accuracy | F1 |
|---|--:|--:|
| Baseline | 0.6494 | 0.0000 |
| Decision Tree | 0.7857 | 0.6916 |
| Random Forest | 0.7338 | 0.5859 |

**The Random Forest loses to a single depth-4 tree, on both metrics.** 300 trees, and it is worse.
That is not a mistake to hide — it is the most interesting line in the table, and a group that says
*"the bigger model lost and here is what we think happened"* scores better than one that quietly
reports only the winner.

### 🟠 Step 5 — Write it up, then get ready to present · *55–60 min*

Two sentences, both short. Nobody is impressed by a high number they cannot account for.

**Then spend the last five minutes on the two things groups always skip.**

1. **Scroll the notebook to the cell you will start talking from**, and run through the four minutes
   once. Hunting for a cell live on the projector costs 20–30 seconds, and ten groups doing that is the
   whole hour.
2. **Ask each other two code questions** — one of each kind from the list above. *"Show me the line
   that handles missing values"* is the one to practise, because it is the one you cannot bluff.
   Two minutes here is the cheapest half-mark in the course.

**What we found:** *(one sentence — what does your score actually mean for the question you asked?)*

**What we do not trust:** *(one limitation — something about the data or the method that would change
the answer)*

**Where we got stuck:** *(if a step defeated you, say which and why — this scores)*

---
## 🟠 Your four minutes on screen

No slides for this round. Open the notebook, scroll, and talk. **240 seconds, and it does not stretch —
ten groups at one minute over is the entire hour gone.**

**Two of you speak, and you hand over in the middle.** Speaker 1 has *what the data says*; speaker 2 has
*what we did about it*. The other members present this same work in the homework video, which happens
every session — so across the six, a group of four gets three turns in the room and three in the video
each, and a group of three gets four and two.

**Five things, in this order. Each one has to be shown on screen, not described.**

| Who | Time | What you say | What must be visible |
|:--:|:--:|---|---|
| **1** | 20s | your topic, and **what one row of this data is** | `df.head()` |
| **1** | 60s | **1 · EDA** — walk the chart you made | **the chart cell, already run** |
| **1** | 40s | **2 · Insight** — what that chart told you | point at the chart or number that shows it |
| ⇄ | | *hand over — say out loud who you are handing to* | |
| **2** | 40s | **3 · Problem** — what the problem with this job is | the cell that shows it |
| **2** | 40s | **4 · Solution** — what you did, **and why that choice** | your `Pipeline` / the fix |
| **2** | 40s | **5 · Metric** — which one, why it fits, **what the baseline scores** | the cell printing the baseline |
| both | +40s | **anyone may be asked about any line of your code** — see below | scroll to the line asked about |
| both | +80s | **3–5 questions from the room**, first hand up | scroll to what they ask about |

> **Saying it is not showing it.** "We made a missing-value chart" without scrolling to it does not
> count. A group that says all five but never opens a real cell scores **0.5, not 1**.

**Problem and Solution are not only about dirty data.** "The question we set cannot be answered with
the columns we have" is a problem, and "so we changed the question to one the data can answer" is a
good solution — if you say why.

**If you trained a model, mention the score inside the metric slot.** There is no separate slot for it,
because the model is not what this round marks.

**Nobody is asked to be flawless.** "We ran out of time at step 4" costs nothing. "We got 0.79 and we do
not know why" costs criterion D.

### 🟠 The 40 seconds of code questions

**The instructor points at one of you — any of you — and asks one question about your own code.** The
mark is the group's and everyone gets the same one, so *"I only wrote the charts"* does not help
anybody. There are two kinds, and you should rehearse both.

**Kind 1 — here is a line, what does it do?**

| Question | What a full answer sounds like |
|---|---|
| what does this line do? | in plain words, not by reading the function name back |
| what breaks if I delete it? | which error, or which number moves and which way |
| why `median` and not `mean`? | tied to your own chart — "our column is right-skewed" |
| what is the difference between `fit` and `transform`? | `fit` learns from train; `transform` applies what it learned |
| how many steps in your `Pipeline`, and what is each one fitted on? | train only — and you should sound certain about that |
| your `baseline()` printed accuracy and F1 far apart. Why? | the classes are imbalanced, so accuracy is the wrong metric here |

**Kind 2 — here is a thing, find it in your code.**

| Question | What you have to do |
|---|---|
| show me the line that deals with missing values | scroll to it, within a few seconds |
| which line splits train and test? | point at `train_test_split`, say the ratio |
| that 0.78 on screen — which line produced it? | point at the exact line |
| where would you change it to use a `RandomForest`? | one place in the `Pipeline` |
| you dropped a column — which command, and why? | point at the `drop`, give the reason |

> **Kind 2 is the one that separates groups.** If AI wrote your code, you can usually still answer
> kind 1 — AI explains lines too. But **"scroll to the line that handles missing values"** needs you to
> have actually read your own file. Ten seconds of hunting is visible from the back of the room.

**Spend two minutes of step 5 asking each other these.** It is the cheapest half-mark in the course.

### 🟠 While the other groups present — you are still working

**Each presenting group takes 3–5 questions from the room, and the first hand up gets the floor.**
You do not have to ask every group. The marks are **yours personally**, not your group's, and they
are on top of the 24:

| Mark | The question |
|:--:|---|
| **1** | **changes what that group should do next** — spots leakage they missed, asks whether their baseline used the right metric, names a limitation they skipped |
| **0.5** | **makes something clearer for the whole room** — why that `k`, show us the cell that number came from |
| **0** | asked to be seen asking · already answered in their four minutes · **or a question you could ask any group without listening** ("why not try another model?") |

**That last one is the real filter.** A question that fits every group means you listened to none.
**Being fast gets you the microphone; it does not get you the mark.**

**Capped at 2 marks per person per session** — once you have two, stop and leave the floor to someone
else. With 30–50 question slots in the hour and about 35 people in the room, there is roughly one turn
each if everybody wants one. **The people who get the marks are the ones who wrote a question down
during the four minutes**, not the ones who started thinking when the hand went up.

You cannot question your own group, and you get one question per presenting group.

### ✅ Before you leave the room

```
[ ] header filled in: GROUP letter · MEMBERS · TOPIC_ID                     must
[ ] IN_CLASS (2 people) + IN_CLIP cover everyone, nobody listed twice       must
[ ] Restart and run all -> runs to the end (if it breaks, write where)

the five things, each with a cell you can scroll to
[ ] 1 EDA       - the chart cell, already run
[ ] 2 Insight   - a finding, not a description of the chart
[ ] 3 Problem   - what is wrong with this job, and the cell that shows it
[ ] 4 Solution  - what you did and why, inside a Pipeline, split before fit
[ ] 5 Metric    - which metric and why, with the baseline on the same metric

[ ] notebook scrolled to where you start talking, four minutes rehearsed
[ ] asked each other 2 code questions - incl. one "show me the line that..."
```

| How far you got | Where that lands |
|---|---|
| nothing to hand in, or did not present | **nothing for this session** |
| presented and showed real cells — even if some of the five are missing or rough | **partial credit, guaranteed** |
| all five, each opened on screen, **and at least one shows real analysis** | **full credit** |

"Real analysis" means one of: the insight **changed what you did next** · the metric's reason is tied
to **your own** data ("only 35% of ours are positive") · you **doubted your own result** and said where.

**Running out of time costs you nothing** as long as you wrote down where you got stuck. The only way to
score nothing is having nothing to show.

**Hand in the notebook as a file** — *File → Download → Download .ipynb*. **Not the Colab link.** A link
can be edited after the hour ends and a file cannot, which is the whole point of freezing it here.

**Not one line mentions a model.** That is deliberate — steps 1 to 3 are the assignment.

### The rest of the mark is the homework

Due **the day before the next session**. It is a *separate* file and a *separate* mark — the notebook you
hand in then does not have to match the one you just presented. Change the method, the model, even the
question, as long as you write down why you changed it. **Changing something because presenting it
showed you a problem is exactly what earns criterion C.**

One catch worth knowing: **the code questions are asked here, in class, about the notebook you just
wrote** — because code you wrote yourself in the last hour, with someone watching, is the hardest thing
to fake. So if your homework rewrites it, slide 13 has to walk through the *homework* code, and expect
to be asked about that instead next week.

Sessions 1–3 add slides and a video; sessions 4–6 are the notebook alone. What the homework carries
that this hour did not: the model finished, **two settings compared**, where the model got it wrong,
three limitations, and **5–10 lines of your own code explained line by line**.

> ### ⛔ Everything is handed in as a file. Links are not accepted.
>
> **No YouTube, no Google Drive, no Google Slides, no Colab link, no file-sharing site.**
>
> | What | Format |
> |---|---|
> | notebook | `.ipynb` |
> | video | `.mp4` (or `.mov`) |
> | slides | `.pdf` or `.pptx` |
> | README | `.md` or `.txt` |
>
> A link can be edited after the deadline, and it can go dead or turn private and leave nothing to mark.
> **A missing file costs you the hand-in mark and everything that had to be watched or read inside it.**

**Two things checked in ten seconds, and lost for nothing if you skip them:** slides are **10 pages or
more**, the video is **5 minutes or more**. Every chart on those slides needs a title, labelled axes and
a caption saying what it shows — plus **at least one concrete example**, a real row or a case the model
got wrong.

---
# Session 1 Wrap-Up

### Four things to remember

1. **A number without a baseline means nothing.** 90% accuracy can mean the model learned nothing at all (C.6)
2. **A result that is too good is a warning, not a win.** The 100% in C.3 was target leakage
3. **Time spent on the data beats time spent on the model.** Preprocessing bought 16.2 points; switching models bought 3.4
4. **A baseline exists for every kind of answer.** Most-common-class for a label, the median for a number — and R² below zero means you lost to it

### What to hand in
- **This notebook** with your hour's work in it, running from the top of the 🟠 section
- The **before/after** preprocessing table, with the baseline beside it
- A `Pipeline` that runs on both titanic and your group's own data
- Your group's chosen topic from the Part 2 list, with `TOPIC_ID` set

### Next session — Regression, and measuring it properly
Today the answer was a label: survived or not. **Next session the answer is a number**, which changes
what "wrong" means — being off by 2 is not the same as being off by 200, and MAE, RMSE and R² each
take a different view of that. Block E was the trailer.